# Design sequences with RL-SAE
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/07_design_with_rl_sae.ipynb)

Use a fixed SAE and feature signature as a reward while adapting the generation policy. Compare feature coverage before and after training, including component scores when combining signatures.

This notebook runs independently. Select **Runtime → Change runtime type → GPU** in Colab.
First use downloads model weights. Training is a small workflow demonstration, not a converged design experiment. GPU memory requirements depend on the model, batch size, and length; a free Colab GPU is not guaranteed to fit RL-SAE.
The setup installs the `v1` release when IDiom is absent. If using an older installation,
upgrade to that release and restart the kernel. No adjacent helper files are required.

In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "idiom[cookbook] @ git+https://github.com/rotskoff-group/idiom.git@v1"])
if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas>=2"])

import json
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from idiom import IDiom, IDiomSAE
from idiom.data.records import Record
from idiom.utils.notebook_helpers import (
    load_inputs, idr_sequence, isolated, check_context, summaries, write_fasta,
    save_run, sequence_metrics, nearest_reference, split_records, example_file,
)
print("Python:", sys.version.split()[0])
started = time.perf_counter()

## Inputs and settings
Run top to bottom. Upload a FASTA using Colab's Files pane and set its path below, or
leave the demo input unchanged. `INPUT_MODE="idr"` accepts ordinary headers for isolated
IDRs; `"annotated"` requires full-protein headers ending in `_IDR_x-y` (1-based inclusive).
Python coordinates are 0-based, end-exclusive. These workflows do not predict IDR boundaries.

For persistent outputs, optionally mount Drive in your own cell with
`from google.colab import drive; drive.mount("/content/drive")`, then set `OUT_DIR` there.
Use a new output directory for each experiment. Rejected records are reported in an audit.

In [ ]:
MODEL_ID = "jxliu2/idiom-300M"
DEVICE = "auto"
BATCH_SIZE = 1
SEED = 0
MAX_STEPS = 10
GROUP_SIZE = 2
LEARNING_RATE = 5e-6
TARGET_LENGTH = 50
MAX_NEW_TOKENS = 96
N = 8
RESUME_FROM = None
OUT_DIR = Path("rl_sae_outputs")

SAE_ID = "jxliu2/idiomsae-300M-L18-k32"
SAE_DEVICE = "cpu" # CPU scoring saves GPU memory but is slower; use "cuda" when memory permits
SIGNATURE_FILE = None # None uses the signature bundled with the package
CASE = "top30"
SIGNATURE_NAMES = ["nucleolus"] # Add a second name to optimize their union

In [ ]:
import gc
import torch
import lightning as L
from importlib.resources import files
from omegaconf import OmegaConf
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
L.seed_everything(SEED, workers=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
if (OUT_DIR / "training").exists() and RESUME_FROM is None:
    raise ValueError("Use a fresh OUT_DIR or set RESUME_FROM to a training checkpoint.")
from idiom.utils.device import resolve_device
training_device = resolve_device(DEVICE)
precision = "bf16-mixed" if training_device.type == "cuda" and torch.cuda.is_bf16_supported() else "32-true"
print("CUDA:", torch.cuda.is_available(), "Training precision:", precision)

## Select feature targets
The default is a bundled nucleolus signature. You can instead upload `signature.json` from
notebook 04. Signatures are specific to SAE weights. The reward uses the frozen original host
and SAE, even while the policy changes; it does not attach the SAE to the trained policy.

For a composite target, add another signature name. The union removes repeated feature IDs.
Each component is evaluated separately below, since a high union score can hide imbalance.
The CPU lens saves GPU memory; RL-SAE still holds a policy, reference policy, and optimizer.

In [ ]:
from idiom.sae.features import load_signatures, combine_signatures, write_signature
from idiom.train.grpo.reward.sae_feature import DEFAULT_FEATURES, sae_signature
signature_source = SIGNATURE_FILE or DEFAULT_FEATURES
lens = IDiomSAE.from_pretrained(SAE_ID, device=SAE_DEVICE)
sets = load_signatures(signature_source, case=CASE, sae=SAE_ID, num_latents=lens.sae.num_latents)
components = {name: sets[name] for name in SIGNATURE_NAMES}
combined = combine_signatures(components)
if "combined" in components:
    raise ValueError("The name 'combined' is reserved by this example.")
training_signatures = write_signature(OUT_DIR / "signature.json", {**components, "combined": combined},
                                      case=CASE, provenance=dict(sae=SAE_ID, components=SIGNATURE_NAMES))
del lens
gc.collect()
reward_spec = dict(name="sae_signature", signature="combined", features=str(training_signatures.resolve()),
                   case=CASE, sae=SAE_ID, device=SAE_DEVICE)
print("Component sizes:", {name: len(ids) for name, ids in components.items()})
print("Union size:", len(combined))

## Generate a baseline
Sample before training, then release the inference model to free memory.

In [ ]:
base = IDiom.from_pretrained(MODEL_ID, device=DEVICE)
sampling = dict(n=N, batch_size=BATCH_SIZE, seed=SEED, temperature=1.0,
                max_new_tokens=MAX_NEW_TOKENS)
baseline = base.generate_unprompted(**sampling)
write_fasta([Record(f"baseline_{i}", s, 0, len(s)) for i, s in enumerate(baseline) if s],
            OUT_DIR / "baseline.fasta")
model_context = base.model.cfg.max_seq_len
del base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Configure GRPO
The objective is the weighted sum of shaped rewards, with a KL penalty relative to the frozen
starting policy. Multiple completions per prompt provide within-group comparisons. These small
settings demonstrate execution; improvement is not guaranteed. Inspect raw component scores too.

In [ ]:
from idiom.train.grpo.train_grpo import build
from idiom.train.grpo.reward import build_reward
from idiom.train.grpo.data import collate_prompts
from torch.utils.data import DataLoader
cfg = OmegaConf.load(files("idiom") / "configs/grpo.yaml")
cfg.init_from = MODEL_ID
cfg.seed = SEED
cfg.prompts.n = max(16, MAX_STEPS * BATCH_SIZE)
cfg.prompts.batch_size = BATCH_SIZE
cfg.grpo.group_size = GROUP_SIZE
cfg.grpo.max_new_tokens = MAX_NEW_TOKENS
cfg.grpo.lr = LEARNING_RATE
cfg.grpo.track_disorder = False
cfg.grpo.log_samples_every = 0
cfg.trainer.max_steps = MAX_STEPS
cfg.out_dir = str(OUT_DIR.resolve())
cfg.resume_from = RESUME_FROM
cfg.device = str(training_device)
terms = [
    dict(label="length", weight=1.0, reward=dict(name="length"),
         shaping=dict(name="quadratic", target=TARGET_LENGTH, width=0.2)),
    dict(label="entropy", weight=1.0, reward=dict(name="entropy"),
         shaping=dict(name="quadratic", target=3.65, width=0.2)),
]

terms.append(dict(label="signature", weight=1.0, reward=reward_spec, shaping=dict(name="identity")))

cfg.reward.terms = terms
cfg.trainer = dict(max_steps=MAX_STEPS, accelerator="gpu" if training_device.type == "cuda" else "cpu",
                   devices=[training_device.index or 0] if training_device.type == "cuda" else 1,
                   precision=precision, gradient_clip_val=1.0, accumulate_grad_batches=1,
                   log_every_n_steps=1, limit_val_batches=2, num_sanity_val_steps=0,
                   enable_model_summary=False)
OmegaConf.save(cfg, OUT_DIR / "training_config.yaml")
lit, prompts = build(cfg)
loader = DataLoader(prompts, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_prompts)

In [ ]:
checkpoint = ModelCheckpoint(dirpath=OUT_DIR / "training/checkpoints", save_last=True,
                             save_top_k=0, every_n_train_steps=max(1, min(10, MAX_STEPS)))
trainer = L.Trainer(**OmegaConf.to_container(cfg.trainer, resolve=True),
                    logger=CSVLogger(OUT_DIR / "training", name="metrics"), callbacks=[checkpoint])
trainer.fit(lit, train_dataloaders=loader, ckpt_path=RESUME_FROM)

## Save, reload, and generate

In [ ]:
trainer.save_checkpoint(OUT_DIR / "training/checkpoints/last.ckpt")
release = OUT_DIR / "model"
IDiom(lit.model.cpu()).save_pretrained(release)
del trainer, lit
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
adapted_model = IDiom.from_pretrained(release, device=DEVICE)
adapted = adapted_model.generate_unprompted(**sampling)
write_fasta([Record(f"adapted_{i}", s, 0, len(s)) for i, s in enumerate(adapted) if s],
            OUT_DIR / "adapted.fasta")
del adapted_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Evaluate the complete objective and sequence quality

In [ ]:
comparison = pd.concat([sequence_metrics(baseline).assign(group="baseline"),
                        sequence_metrics(adapted).assign(group="adapted")], ignore_index=True)
comparison.to_csv(OUT_DIR / "candidates.csv", index=False)
display(comparison.groupby("group").agg(count=("sequence", "size"), mean_length=("length", "mean"),
                                        mean_entropy=("entropy", "mean"), duplicate_fraction=("duplicate", "mean")))
fig, axes = plt.subplots(1, 2, figsize=(8, 3), constrained_layout=True)
for group, rows in comparison.groupby("group"):
    axes[0].hist(rows.length, bins=10, alpha=0.5, label=group)
    axes[1].hist(rows.entropy, bins=10, alpha=0.5, label=group)
axes[0].set(xlabel="Length", ylabel="Count")
axes[1].set(xlabel="Composition entropy (bits)")
axes[0].legend()
fig.savefig(OUT_DIR / "comparison.png", dpi=160)
plt.show()

objective = build_reward(cfg.reward)
for group, sequences in (("baseline", baseline), ("adapted", adapted)):
    totals, details = objective(sequences, group_size=1)
    scores = pd.DataFrame(details)
    scores["total_reward"] = totals
    scores["sequence"] = sequences
    scores.to_csv(OUT_DIR / f"{group}_rewards.csv", index=False)
    print(group)
    display(scores)

## Inspect feature coverage and component balance
Use the same frozen-lens scoring function as training. Coverage is the fraction of target features
with positive activation anywhere in an IDR, not a prediction of biological function.
The heatmap below records individual feature presence in generated sequences.

In [ ]:
component_rows = []
for name in [*SIGNATURE_NAMES, "combined"]:
    score = sae_signature(name, features=str(training_signatures.resolve()), case=CASE,
                          sae=SAE_ID, device=SAE_DEVICE)
    for group, sequences in (("baseline", baseline), ("adapted", adapted)):
        for i, value in enumerate(score(sequences)):
            component_rows.append(dict(group=group, sequence_id=i, component=name, coverage=value))
coverage = pd.DataFrame(component_rows)
coverage.to_csv(OUT_DIR / "signature_coverage.csv", index=False)
display(coverage.groupby(["group", "component"]).coverage.mean().unstack())
from idiom.train.grpo.reward.sae_feature import signature_presence
nonempty = [s for s in adapted if s]
if nonempty:
    presence = signature_presence(nonempty, "combined", features=str(training_signatures.resolve()),
                                  case=CASE, sae=SAE_ID, device=SAE_DEVICE)
    pd.DataFrame(presence, columns=combined).to_csv(OUT_DIR / "target_feature_presence.csv", index=False)
    fig, ax = plt.subplots(figsize=(9, 3), constrained_layout=True)
    ax.imshow(presence, aspect="auto", vmin=0, vmax=1, cmap="Oranges")
    ax.set_xticks(range(len(combined)), combined, rotation=90, fontsize=6)
    ax.set(xlabel="Target feature", ylabel="Nonempty generated sequence")
    fig.savefig(OUT_DIR / "target_feature_presence.png", dpi=160)
    plt.show()

In [ ]:
save_run(OUT_DIR, dict(model=MODEL_ID, seed=SEED, steps=MAX_STEPS, sampling=sampling,
                       reward=OmegaConf.to_container(cfg.reward, resolve=True), resume_from=RESUME_FROM),
         elapsed=time.perf_counter() - started)

## Save and continue
`model/` is a reloadable release; `training/checkpoints/last.ckpt` also preserves optimizer
state for resuming. `training/metrics/` contains CSV training logs. Keep the training config,
input audit, and run settings with generated sequences. These short runs demonstrate the workflow;
assess held-out data, diversity, and independent measurements before drawing design conclusions.
The download includes checkpoints and can be large; use Drive for long-running experiments.

In [ ]:
import shutil
archive = shutil.make_archive(str(OUT_DIR.resolve()), "zip", OUT_DIR)
print("Results:", OUT_DIR.resolve(), "\nDownload:", archive)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(archive)